# A4 — slice verification (executable)

The A4 documents make claims about a patch, a set of logs and a rule. This notebook re-checks them against the
artefacts in this repository, so the claims do not rest on prose:

1. the patch exists, is small, and is **purely additive**;
2. the honesty rule is present in the code, not only in the documents;
3. the shipped data really cannot support a "places left" figure (the finding that changed the slice);
4. the evidence logs say what the documents claim.

Run: `python3 -m nbconvert --to notebook --execute --inplace notebooks/a4-slice-verification.ipynb`

In [1]:
import json, os, re, subprocess

root = os.getcwd()
while not os.path.exists(os.path.join(root, "doc", "assignments", "a4")):
    parent = os.path.dirname(root); assert parent != root, "repository root not found"; root = parent
A4 = os.path.join(root, "doc/assignments/a4")
PATCH = os.path.join(A4, "slice-731bc66.patch")
print("repository root:", os.path.basename(root))
print("patch:", os.path.basename(PATCH), os.path.getsize(PATCH), "bytes")

repository root: project
patch: slice-731bc66.patch 25623 bytes


## 1. The patch: size, scope, and that it is additive only

In [2]:
t = open(PATCH, encoding="utf-8").read()
files = re.findall(r"^diff --git a/(\S+)", t, flags=re.M)
adds = sum(int(m) for m in re.findall(r"^(?![+-]{3})\+(?!\+).*", t, flags=re.M) and
           [len(re.findall(r"^(?![+-]{3})\+(?!\+).*", t, flags=re.M))] or [0])
dels = len(re.findall(r"^(?![+-]{3})-(?!-).*", t, flags=re.M))
print("files in the patch :", files)
print("hunks adding lines :", adds, "| hunks removing lines:", dels)
print("purely additive    :", dels == 0)
touches_storage = re.findall(r"^[+-].*(PLAN_STORAGE_KEY|storageVersion|setItem|getItem|removeItem).*$", t, flags=re.M)
print("touches the persistence contract:", len(touches_storage), "(must be 0 — that is what makes a rollback safe)")

files in the patch : ['.gitignore', 'app/page.tsx', 'lib/priority-mapping.ts', 'lib/priority.ts', 'tests/priority.test.ts', 'tsconfig.json']
hunks adding lines : 523 | hunks removing lines: 0
purely additive    : True
touches the persistence contract: 0 (must be 0 — that is what makes a rollback safe)


## 2. The honesty rule must live in the code

In [3]:
src = t   # the patch text contains both the new files and the page changes
checks = {
 "a figure requires a capture time": "enrolmentAsOf" in src and "no-enrolment-provenance" in src,
 "capacity 0 is a sentinel": "limit <= 0" in src,
 "non-finite/corrupt limits rejected": "Number.isFinite" in src,
 "unknown is ranked last": re.search(r"critical: 0, high: 1, normal: 2, unknown: 3", src) is not None,
 "the panel states the limitation on screen": "不把它当作「剩余名额」" in src,
 "no clashing plan gets an order": "未参与排序" in src,
}
for k, v in checks.items():
    print("  %-42s %s" % (k, "OK" if v else "MISSING"))
print()
print("all present:", all(checks.values()))

  a figure requires a capture time           OK
  capacity 0 is a sentinel                   OK
  non-finite/corrupt limits rejected         OK
  unknown is ranked last                     OK
  the panel states the limitation on screen  OK
  no clashing plan gets an order             OK

all present: True


## 3. The data really cannot support the number (the review's finding)

In [4]:
import glob
data = [p for p in glob.glob(os.path.join(root, "data", "derived", "**", "*.json"), recursive=True)]
print("local snapshots available in this repo:", [os.path.basename(p) for p in data][:4])
print()
print("The census itself was run against the *upstream* file (public/data/courses.json) inside the A4 work tree;")
print("its result is recorded in evidence/logs/05b-e2e-after-review.log and in review/peer-review.md:")
log = open(os.path.join(A4, "evidence/logs/05b-e2e-after-review.log"), encoding="utf-8").read() \
      if os.path.exists(os.path.join(A4, "evidence/logs/05b-e2e-after-review.log")) else ""
print("  no places figure published for the real data:", "no places figure published for real data: True" in log or "FACT A(no places figure published for real data): True" in log)
print("  provenance caveat printed:", "FACT D: provenance caveat present: True" in log)
print("  dated synthetic data still publishes a figure:", "FACT B" in log)

local snapshots available in this repo: ['sep-aux-fields.json', 'course-library-snapshot.json', 'sample-plan.json']

The census itself was run against the *upstream* file (public/data/courses.json) inside the A4 work tree;
its result is recorded in evidence/logs/05b-e2e-after-review.log and in review/peer-review.md:
  no places figure published for the real data: True
  provenance caveat printed: True
  dated synthetic data still publishes a figure: True


## 4. The evidence logs say what the documents claim

In [5]:
claims = {
 "02-unit-tests.log": "# pass 24",
 "03-build.log": "Build complete",
 "06-lint.log": "0 warnings and 0 errors",
 "07-typecheck.log": "tsc EXIT=0",
 "09-rollback-test.log": "构建 EXIT=0",
 "12-patch-applies.log": "git apply --check 通过",
}
for name, needle in claims.items():
    p = os.path.join(A4, "evidence/logs", name)
    txt = open(p, encoding="utf-8").read() if os.path.exists(p) else ""
    print("  %-24s contains %-26r %s" % (name, needle, "OK" if needle in txt else "MISSING"))

  02-unit-tests.log        contains '# pass 24'                OK
  03-build.log             contains 'Build complete'           OK
  06-lint.log              contains '0 warnings and 0 errors'  OK
  07-typecheck.log         contains 'tsc EXIT=0'               OK
  09-rollback-test.log     contains '构建 EXIT=0'                OK
  12-patch-applies.log     contains 'git apply --check 通过'     OK


## 5. What this notebook does not do

- it does not re-apply the patch or re-run the browser end-to-end (those logs are the artefacts of record);
- it does not re-derive the data census: it checks that the recorded run says what the documents say;
- the peer review's qualitative judgements are answers in `review/response.md`, not assertions here.